In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# ============================================================
# LOAD BEST CONFIG
# ============================================================

import json

# Open the file and parse its contents
with open('descriptornet_supervised_bho/best_config.json', 'r') as file:
    best_configuration = json.load(file)
best_configuration

{'depth': 3,
 'width': 128,
 'activation': 'tanh',
 'lr': 0.0037600695879154403,
 'batch_size': 16,
 'weight_decay': 0.001}

In [3]:
# ============================================================
# CONFIG
# ============================================================
DATA_DIR    = "../plga_dataset/release_dataset_with_Crank_release_even.xlsx"
NET2_WEIGHTS = "../physicsnet/physicsnet_pretrained.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Network 1 (trained) ---
N1_DEPTH = best_configuration["depth"]
N1_WIDTH = best_configuration["width"]
N1_ACT = best_configuration["activation"]

# --- Network 2 (frozen, must match pretrain_net2.py exactly) ---
N2_DEPTH = 3
N2_WIDTH = 256
N2_ACT = "silu"

# --- Training ---
EPOCHS = 2500
LR = best_configuration["lr"]
BATCH_SIZE = best_configuration["batch_size"]
WEIGHT_DECAY = best_configuration["weight_decay"]
N_RHO_INTEGRATION = 30

# --- Data ---
DESCRIPTOR_COLS = [
    'Drug MW', 'Drug TPSA', 'Drug LogP', 'Polymer MW', 'LA/GA',
    'Initial Drug-to-Polymer Ratio', 'Particle Size',
    'Drug Loading Capacity', 'Drug Encapsulation Efficiency',
    'Solubility Enhancer Concentration'
]
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

In [4]:
# ============================================================
# DATA LOADING
# ============================================================
def load_data(path):
    """
    One row per formulation: descriptors -> Crank-fitted D.
    No release profiles needed — this is direct supervised regression.
    """
    df = pd.read_excel(path)
    df = df.drop(columns=['Time', 'Release', 'Crank_Release'], errors='ignore')
    df = df.drop_duplicates(subset=['Formulation Index']).reset_index(drop=True)

    fid = df['Formulation Index'].values
    X   = df.drop(columns=['Formulation Index', 'Crank_D']).values.astype(float)
    D   = df['Crank_D'].values.astype(float)

    print(f"Formulations: {len(fid)} | Descriptor columns: {X.shape[1]}")
    print(f"D range: [{D.min():.3e}, {D.max():.3e}]")

    return X, D, fid


# ============================================================
# NETWORK
# ============================================================
def get_activation(name):
    return {"tanh": nn.Tanh(), "silu": nn.SiLU(), "gelu": nn.GELU()}[name.lower()]


class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, depth, width, activation):
        super().__init__()
        layers = [nn.Linear(in_dim, width), get_activation(activation)]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), get_activation(activation)]
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class CrankDPredictor(nn.Module):
    """Descriptors -> log10(D). Direct supervised regression, no physics."""
    def __init__(self, n_desc, depth, width, activation):
        super().__init__()
        self.mlp = MLP(n_desc, 1, depth, width, activation)

    def forward(self, x):
        return self.mlp(x)   # (B, 1) — log10(D)


# ============================================================
# EVALUATION
# ============================================================
@torch.no_grad()
def eval_mse_male(model, X_t, y_log_t):
    """
    X_t      : (N, n_desc) descriptors, scaled
    y_log_t  : (N, 1) true log10(D)
    Returns (MSE on log10(D), MALE) 
    """
    model.eval()
    pred_log = model(X_t)
    mse  = torch.mean((pred_log - y_log_t) ** 2).item()
    male = torch.mean(torch.abs(pred_log - y_log_t)).item()
    
    if not np.isfinite(mse):
        mse = float("inf")
    if not np.isfinite(male):
        male = float("inf")
        
    return mse, male

In [5]:
# ============================================================
# EVALUATION
# ============================================================
@torch.no_grad()
def eval_mse_male(model, X_t, y_log_t):
    """
    X_t      : (N, n_desc) descriptors, scaled
    y_log_t  : (N, 1) true log10(D)
    Returns (MSE on log10(D), MALE) — for this baseline they coincide
    in spirit, but MSE is squared-error (used for selection) and MALE
    is mean absolute error (used for reporting/comparison with PINN).
    """
    model.eval()
    pred_log = model(X_t)
    mse  = torch.mean((pred_log - y_log_t) ** 2).item()
    male = torch.mean(torch.abs(pred_log - y_log_t)).item()
    if not np.isfinite(mse):
        mse = float("inf")
    if not np.isfinite(male):
        male = float("inf")
    return mse, male

In [6]:
# ============================================================
# MAIN
# ============================================================
def main():
    test_error_list = []
    print(f"Device: {DEVICE}\n")

    # --- Load data ---
    try:
        (X, D, fid) = load_data(DATA_DIR)
    except FileNotFoundError:
        print(f"Error: Could not find {DATA_DIR}. Please check the path.")
        return

    n_desc = X.shape[1]
    print(f"\n{n_desc} descriptor columns.\n")

    # Log10 transformation for regression targets
    y_logD = np.log10(D)

    # Train/test split by FORMULATION
    all_fids = np.unique(fid)
    fids_train, fids_test = train_test_split(
        all_fids, test_size=0.2, random_state=SEED
    )

    train_mask = np.isin(fid, fids_train)
    test_mask  = np.isin(fid, fids_test)

    # Descriptor scaler fit on training rows only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X[train_mask])
    X_test_scaled  = scaler.transform(X[test_mask])

    # Convert to Tensors
    X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32, device=DEVICE)
    y_train_t = torch.tensor(y_logD[train_mask], dtype=torch.float32, device=DEVICE).unsqueeze(1)
    
    X_test_t  = torch.tensor(X_test_scaled, dtype=torch.float32, device=DEVICE)
    y_test_t  = torch.tensor(y_logD[test_mask], dtype=torch.float32, device=DEVICE).unsqueeze(1)

    # DataLoaders 
    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    print(f"Train samples: {train_mask.sum()}")
    print(f"Test samples:  {test_mask.sum()}")

    # --- Networks ---
    net1 = CrankDPredictor(n_desc=n_desc, depth=N1_DEPTH, width=N1_WIDTH, activation=N1_ACT).to(DEVICE)
    
    # Optional: Bias initialization (if you need the network to start predicting near log10_D_init)
    log10_D_init = -10.0
    with torch.no_grad():
        net1.mlp.net[-1].bias.fill_(log10_D_init)
        
    print(f"\nNetwork 1 bias init: log10(D) = {log10_D_init:.3f}")
    print(f"Network 1 params: {sum(p.numel() for p in net1.parameters()):,}")

    # --- Training Setup ---
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(net1.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    print("\nStarting Training...")
    print("-" * 70)
    
    for epoch in range(EPOCHS):
        net1.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            preds = net1(batch_X)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()

        # Evaluation phase
        test_mse, test_male = eval_mse_male(net1, X_test_t, y_test_t)
        test_error_list.append(test_mse)

        if epoch % 1 == 0 or epoch == EPOCHS - 1:
            print(f"Epoch [{epoch:4d}/{EPOCHS}] | Test MSE: {test_mse:.4f} | Test MALE: {test_male:.4f}")

    print("-" * 70)
    best_err = min(test_error_list)
    best_idx = test_error_list.index(best_err)
    print(f"Min_error = {best_err:.4f}, achieved at epoch {best_idx}")
    
    return net1

## Train until 2500 iterations

In [ ]:
SEED = 0
EPOCHS = 2500
torch.manual_seed(SEED)
np.random.seed(SEED)
descriptornet = main()

## Train with early stopping - until best recorded performance

In [ ]:
SEED = 0
EPOCHS = 20
torch.manual_seed(SEED)
np.random.seed(SEED)
descriptornet = main()

## Performance with test dataset

In [9]:
(X, D, fid) = load_data(DATA_DIR)

n_desc = X.shape[1]
print(f"\n{n_desc} descriptor columns.\n")

# Log10 transformation for regression targets
y_logD = np.log10(D)

# Train/test split by FORMULATION
all_fids = np.unique(fid)
fids_train, fids_test = train_test_split(
    all_fids, test_size=0.2, random_state=SEED
)

train_mask = np.isin(fid, fids_train)
test_mask  = np.isin(fid, fids_test)

# Descriptor scaler fit on training rows only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X[train_mask])
X_test_scaled  = scaler.transform(X[test_mask])

# Convert to Tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_logD[train_mask], dtype=torch.float32, device=DEVICE).unsqueeze(1)
    
X_test_t  = torch.tensor(X_test_scaled, dtype=torch.float32, device=DEVICE)
y_test_t  = torch.tensor(y_logD[test_mask], dtype=torch.float32, device=DEVICE).unsqueeze(1)

Formulations: 321 | Descriptor columns: 10
D range: [7.181e-21, 2.736e-15]

10 descriptor columns.



In [10]:
y_pred = descriptornet(X_test_t)

y_true = y_test_t.squeeze().detach().cpu().numpy()
y_pred = y_pred.squeeze().detach().cpu().numpy()

In [11]:
male = np.mean(np.abs(y_true - y_pred))
mse = np.mean((y_true - y_pred)**2)

In [12]:
mse, male

(np.float32(0.14673597), np.float32(0.30975097))

In [ ]:
def parity_plot(y_true, y_pred, lo=-15.5, hi=-10.5, save=None):
    """Parity plot for log10(D_eff). Shared across Figs 6, 8, 9, 11."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    r2 = r2_score(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(3.4, 3.4), constrained_layout=True)

    ax.plot([lo, hi], [lo, hi], color='gray', linestyle='--',
            zorder=1, label=r'$y = x$')                       # identity first, behind
    ax.scatter(y_true, y_pred, color='#1f77b4', alpha=0.7,
               edgecolor='black', linewidth=0.4, s=28,
               zorder=2, label='Predictions')

    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect('equal', adjustable='box')                  # true 45° line
    ax.set_xlabel(r"Crank-fitted $\log_{10}(D_{\mathrm{eff}})$")
    ax.set_ylabel(r"Predicted $\log_{10}(D_{\mathrm{eff}})$")
    ax.grid(True)

    ax.text(0.05, 0.95, fr"$R^2 = {r2:.4f}$", transform=ax.transAxes,
            va='top', ha='left', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7', lw=0.6))
    ax.legend(loc='lower right')

    if save:
        fig.savefig(save + ".pdf")
        fig.savefig(save + ".png", dpi=600)
    return fig, ax


# ---- Figure 11: supervised baseline (convert m²/s → cm²/s) ----
# D_cm2s = D_m2s * 1e4  →  log10 shifts by +4
y_true_cm = np.asarray(y_true).ravel() + 4.0
y_pred_cm = np.asarray(y_pred).ravel() + 4.0

parity_plot(y_true_cm, y_pred_cm, save="../plots/fig11_supervised_parity")